# Tutorial 4b: Regresión con SVR, Random Forest, Gradient Boosting y XGBoost

## Introducción

En este tutorial vamos a practicar modelos de **regresión** usando el dataset **Diabetes** de `scikit-learn`. Es un dataset pequeño y muy conocido que contiene 442 pacientes con 10 variables clínicas (edad, IMC, presión arterial, etc.), y la variable objetivo es una medida cuantitativa de la progresión de la diabetes un año después del inicio del estudio.

Vamos a entrenar y comparar los siguientes modelos de regresión:

1. **Support Vector Regression (SVR)**
2. **Random Forest Regressor**
3. **Gradient Boosting Regressor**
4. **XGBoost Regressor**
5. Finalmente, usaremos **PyCaret** para automatizar la comparación.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
%matplotlib inline

np.random.seed(42)

## 2. Carga del dataset Diabetes

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()

# Convertimos a DataFrame
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='progresion')

print("Forma de X:", X.shape)
X.head()

In [ ]:
# Estadísticas básicas de la variable objetivo
print("Estadísticas de la variable objetivo:")
print(y.describe())

plt.figure(figsize=(8,4))
plt.hist(y, bins=30, color='steelblue', edgecolor='black')
plt.title('Distribución de la variable objetivo (progresión de la diabetes)')
plt.xlabel('Progresión')
plt.ylabel('Frecuencia')
plt.show()

### Correlación entre variables
Veamos qué variables están más correlacionadas con la variable objetivo.

In [ ]:
df_corr = X.copy()
df_corr['progresion'] = y

plt.figure(figsize=(10,7))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación')
plt.show()

## 3. División en entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Tamaño de entrenamiento:", X_train.shape)
print("Tamaño de prueba:", X_test.shape)

## 4. Estandarización

Igual que en clasificación, los SVR son sensibles a la escala de las variables, así que estandarizamos.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## 5. Función auxiliar para evaluar modelos

Definimos una función que calcula las métricas más comunes de regresión: **MAE**, **MSE**, **RMSE** y **R²**.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar_modelo(nombre, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {nombre} ---")
    print(f"MAE : {mae:.3f}")
    print(f"MSE : {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²  : {r2:.3f}")
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

## 6. Modelo 1: Support Vector Regression (SVR)

El **SVR** es la versión para regresión del SVM. Busca una función que se ajuste a los datos dentro de un margen de tolerancia $\epsilon$.

In [ ]:
from sklearn.svm import SVR

mdl_svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=1.0)
mdl_svr.fit(X_train_sc, y_train)

y_pred_svr = mdl_svr.predict(X_test_sc)
res_svr = evaluar_modelo("SVR", y_test, y_pred_svr)

## 7. Modelo 2: Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

mdl_rf = RandomForestRegressor(n_estimators=200, random_state=42)
mdl_rf.fit(X_train, y_train)

y_pred_rf = mdl_rf.predict(X_test)
res_rf = evaluar_modelo("Random Forest", y_test, y_pred_rf)

In [ ]:
# Importancia de las variables
def plot_feature_importances(model, feature_names):
    importances = model.feature_importances_
    indices = np.argsort(importances)
    plt.figure(figsize=(8,5))
    plt.title('Importancia de las variables')
    plt.barh(range(len(indices)), importances[indices], color='steelblue', align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('Importancia relativa')
    plt.tight_layout()
    plt.show()

plot_feature_importances(mdl_rf, X.columns.tolist())

In [ ]:
#Implementación del Decision Tree Classifier y visualización
mdl_dt = DecisionTreeRegressor(random_state=42)
mdl_dt.fit(X_train, y_train)

y_pred_dt = mdl_dt.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred_dt))
plt.figure(figsize=(12, 6))
plot_tree(mdl_dt, filled=True, feature_names=X.columns, rounded=True)
plt.show()

## 8. Modelo 3: Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

mdl_gbm = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
mdl_gbm.fit(X_train, y_train)

y_pred_gbm = mdl_gbm.predict(X_test)
res_gbm = evaluar_modelo("Gradient Boosting", y_test, y_pred_gbm)

In [ ]:
plot_feature_importances(mdl_gbm, X.columns.tolist())

## 9. Modelo 4: XGBoost Regressor

In [ ]:
# Si no tienes xgboost instalado: !pip install xgboost
from xgboost import XGBRegressor

mdl_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
mdl_xgb.fit(X_train, y_train)

y_pred_xgb = mdl_xgb.predict(X_test)
res_xgb = evaluar_modelo("XGBoost", y_test, y_pred_xgb)

In [ ]:
plot_feature_importances(mdl_xgb, X.columns.tolist())

## 10. Comparación de modelos

Juntamos los resultados de los 4 modelos en una tabla.

In [ ]:
resultados = pd.DataFrame([res_svr, res_rf, res_gbm, res_xgb])
resultados.sort_values('R2', ascending=False)

In [ ]:
# Visualización: predicciones vs valores reales para el mejor modelo
plt.figure(figsize=(7,7))
plt.scatter(y_test, y_pred_gbm, alpha=0.6, label='Gradient Boosting')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Ideal')
plt.xlabel('Valor real')
plt.ylabel('Predicción')
plt.title('Predicciones vs valores reales')
plt.legend()
plt.show()

## 11. Automatización con PyCaret

Igual que en el notebook de clasificación, usaremos **PyCaret** para automatizar el entrenamiento y la comparación. Aquí solo veremos `setup`, `compare_models` y la obtención de métricas con `pull`.

> **Nota:** Si no tienes PyCaret instalado, ejecuta `!pip install pycaret`.

In [ ]:
# Importamos el módulo de regresión de PyCaret
from pycaret.regression import setup, compare_models, pull,plot_model,predict_model

# Preparamos el dataframe completo
df_pycaret = X.copy()
df_pycaret['progresion'] = y

In [ ]:
# Configuramos el experimento
exp = setup(data=df_pycaret, target='progresion', session_id=42, verbose=False)

In [ ]:
# Comparamos varios modelos de regresión automáticamente
best_model = compare_models(n_select=1,sort="MAE")

In [ ]:
# Obtenemos la tabla con las métricas
metricas = pull()
metricas

In [ ]:
plot_model(best_model, plot = 'residuals')

In [ ]:
plot_model(best_model, plot = 'error')

In [ ]:
plot_model(best_model, plot = 'feature')

In [ ]:
holdout_pred = predict_model(best_model)

### Conclusiones

* Practicamos 4 modelos de regresión: **SVR, Random Forest, Gradient Boosting y XGBoost**.
* Las métricas usadas en regresión son distintas a las de clasificación: **MAE, MSE, RMSE y R²**.
* El dataset Diabetes es difícil: con sólo 442 muestras y 10 variables, los R² rondan 0.4–0.5, lo cual indica que las variables explican parcialmente la progresión de la enfermedad.
* **PyCaret** nos permite comparar rápidamente muchos modelos sin tener que escribir código repetitivo para cada uno.

### Ejercicio
1. Ajusta los hiperparámetros de cada modelo (por ejemplo `max_depth`, `learning_rate`, `n_estimators`) y observa cómo cambian las métricas.